# SAC Drone Navigation

This notebook contains the SAC setup, curriculum training code and results for the Easy, Medium and Hard environments.

In [ ]:
# Training settings
TRAIN_FULL = False
EASY_TIMESTEPS = 1_000_000
MEDIUM_TIMESTEPS = 2_000_000
HARD_TIMESTEPS = 3_000_000
N_ENVS = 4
SEED = 0
DEVICE = "cuda"
EVALUATION_EPISODES = 20
SAVED_MODEL_EPISODES = 3

LEVELS = {
    "easy": EASY_TIMESTEPS,
    "medium": MEDIUM_TIMESTEPS,
    "hard": HARD_TIMESTEPS,
}

In [ ]:
import sys
import time
from datetime import datetime
from pathlib import Path

SUBMISSION_DIR = Path.cwd()
if not (SUBMISSION_DIR / "models").exists():
    SUBMISSION_DIR = SUBMISSION_DIR / "sac_submission"
PROJECT_ROOT = SUBMISSION_DIR.parent
sys.path.insert(0, str(SUBMISSION_DIR))
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv

from reward_modify.runtime import SeededNavigate, SACNavigateExtractor

ROOT = SUBMISSION_DIR
MODEL_DIR = ROOT / "models"
OUTPUT_ROOT = ROOT / "outputs"
ACTUAL_DEVICE = "cuda" if DEVICE == "cuda" and torch.cuda.is_available() else "cpu"

print("Device:", ACTUAL_DEVICE)
print("Levels:", LEVELS)

## 2. Environment and SAC model

In [ ]:
def make_env(level, seed):
    def create():
        return Monitor(SeededNavigate(level, seed))
    return create


def make_vector_env(level, seed):
    env_functions = [make_env(level, seed + i) for i in range(N_ENVS)]
    if sys.platform.startswith("linux"):
        return SubprocVecEnv(env_functions, start_method="fork")
    return DummyVecEnv(env_functions)


def create_model(env, output_dir):
    policy_kwargs = {
        "features_extractor_class": SACNavigateExtractor,
        "features_extractor_kwargs": {"cnn_output_dim": 32},
        "normalize_images": False,
        "net_arch": [128, 128],
        "share_features_extractor": False,
    }

    return SAC(
        "MultiInputPolicy",
        env,
        learning_rate=3e-4,
        buffer_size=50_000,
        learning_starts=2_000,
        batch_size=64,
        train_freq=8,
        gradient_steps=1,
        gamma=0.99,
        tau=0.005,
        seed=SEED,
        device=ACTUAL_DEVICE,
        policy_kwargs=policy_kwargs,
        tensorboard_log=str(output_dir / "tensorboard"),
        verbose=1,
    )

## 3. Evaluation function

In [ ]:
def evaluate_model(model, level, episodes=20, seed=10_000):
    env = SeededNavigate(level, seed)
    successes = 0
    collisions = 0
    rewards = []

    try:
        for episode in range(episodes):
            observation, _ = env.reset(seed=seed + episode)
            done = False
            total_reward = 0.0
            info = {}

            while not done:
                action, _ = model.predict(observation, deterministic=True)
                observation, reward, terminated, truncated, info = env.step(action)
                total_reward += float(reward)
                done = terminated or truncated

            successes += int(info.get("success", False))
            collisions += int(info.get("collision", False))
            rewards.append(total_reward)
    finally:
        env.close()

    return {
        "level": level,
        "episodes": episodes,
        "success_rate": successes / episodes,
        "collision_rate": collisions / episodes,
        "mean_reward": sum(rewards) / episodes,
    }

## 4. Curriculum training

The same SAC model is trained in order: Easy, Medium, then Hard. Training is disabled by default because the trained models are included.

In [ ]:
if TRAIN_FULL:
    run_name = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = OUTPUT_ROOT / run_name
    output_dir.mkdir(parents=True, exist_ok=False)

    model = None
    results = []

    for level_index, (level, timesteps) in enumerate(LEVELS.items()):
        print(f"\nTraining {level}: {timesteps:,} transitions")
        env = make_vector_env(level, SEED + level_index * 1000)
        started = time.time()

        if model is None:
            model = create_model(env, output_dir)
        else:
            model.set_env(env)

        model.learn(total_timesteps=timesteps, reset_num_timesteps=False, tb_log_name=level)
        model.save(output_dir / f"sac_{level}_final")

        result = evaluate_model(
            model,
            level,
            episodes=EVALUATION_EPISODES,
            seed=20_000 + level_index * 1000,
        )
        result["training_minutes"] = (time.time() - started) / 60
        results.append(result)
        env.close()
        display(pd.DataFrame(results))

    model.save(output_dir / "sac_curriculum_final")
    pd.DataFrame(results).to_csv(output_dir / "evaluation.csv", index=False)
    print("Saved to:", output_dir)
else:
    print("Full training is disabled. Set TRAIN_FULL = True to train again.")

## 5. Training results

The table and graph show the evaluation result of the selected model for each level.

In [ ]:
results_path = ROOT / "sac_results" / "evaluation_summary.csv"
recorded_results = pd.read_csv(results_path)
display(recorded_results)

ax = recorded_results.plot(
    x="level",
    y=["success_rate", "collision_rate"],
    kind="bar",
    ylim=(0, 1),
    rot=0,
    figsize=(7, 4),
)
ax.set_ylabel("Rate")
ax.set_title("SAC evaluation results")
plt.tight_layout()
plt.show()

## 6. Test the saved models

This section loads and evaluates the saved Easy, Medium and Hard models.

In [ ]:
saved_model_results = []

for index, level in enumerate(LEVELS):
    model_path = MODEL_DIR / f"best_sac_{level}.zip"
    if not model_path.exists():
        print("Model not found:", model_path)
        continue

    saved_model = SAC.load(model_path, device=ACTUAL_DEVICE)
    result = evaluate_model(
        saved_model,
        level,
        episodes=SAVED_MODEL_EPISODES,
        seed=30_000 + index * 1000,
    )
    saved_model_results.append(result)

display(pd.DataFrame(saved_model_results))